# Orthogonality in High Dimensions

**Description:** Explore why random vectors become nearly orthogonal in high dimensions, how optimization can push vectors apart, and how this geometry relates to the Johnson–Lindenstrauss lemma.
**Level:** Beginner
**Tags:** Interpretability, Linear Algebra, High-Dimensional Geometry, Johnson–Lindenstrauss

High-dimensional vector spaces appear throughout machine learning: embeddings, hidden states, model weights, and activation directions are all vectors. Their geometry is surprisingly different from the geometry we can draw in two or three dimensions.

## Questions we will answer

1. At what angles do two random vectors meet?
2. What changes as the number of dimensions grows?
3. What is the optimization code in the prompt actually doing?
4. Why can many vectors be *nearly* orthogonal, but not exactly orthogonal?
5. How is this related to the Johnson–Lindenstrauss lemma?

## Angles and dot products

For nonzero vectors $x$ and $y$,

$$
\cos(\theta) = \frac{x \cdot y}{\lVert x \rVert\,\lVert y \rVert}.
$$

After normalizing both vectors to length 1, their dot product is their cosine:

- $x \cdot y = 1$: same direction ($0^\circ$)
- $x \cdot y = 0$: orthogonal ($90^\circ$)
- $x \cdot y = -1$: opposite directions ($180^\circ$)

We generate random directions by sampling Gaussian coordinates and normalizing. The resulting directions are uniform on the unit sphere.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(7)

In [ ]:
def random_unit_vectors(count, dimension, rng):
    vectors = rng.normal(size=(count, dimension))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def sample_cosines(dimension, pair_count=30_000):
    x = random_unit_vectors(pair_count, dimension, rng)
    y = random_unit_vectors(pair_count, dimension, rng)
    return np.sum(x * y, axis=1)

## Random angles concentrate near 90°

The next plot samples independent pairs of random directions. In low dimensions, their angles vary widely. As dimension increases, the distribution becomes sharply concentrated around $90^\circ$.

In [ ]:
dimensions = [2, 10, 100, 500]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for dimension, axis in zip(dimensions, axes.flat):
    cosines = sample_cosines(dimension)
    angles = np.degrees(np.arccos(np.clip(cosines, -1, 1)))
    axis.hist(angles, bins=60, range=(0, 180), density=True)
    axis.axvline(90, color="black", linestyle="--", linewidth=1)
    axis.set(title=f"dimension = {dimension}", xlabel="angle (degrees)", ylabel="density")

fig.suptitle("Angles between independent random unit vectors")
fig.tight_layout()

## How the dimension controls the spread

For independent random unit vectors in $d$ dimensions,

$$
\mathbb{E}[\cos(\theta)] = 0, \qquad \operatorname{Var}[\cos(\theta)] = \frac{1}{d}.
$$

The exact angle density is proportional to $\sin^{d-2}(\theta)$ on $0 \leq \theta \leq \pi$. As $d$ grows, this puts almost all its mass near $\pi/2$. Equivalently, the typical cosine has size about $1/\sqrt{d}$. Near $90^\circ$, the angle's standard deviation is approximately $1/\sqrt{d}$ radians, or $57.3/\sqrt{d}$ degrees. Doubling the dimension does **not** halve the spread; multiplying the dimension by four approximately halves it.

In [ ]:
print(f"{'dimension':>10} {'measured angle std':>20} {'57.3 / sqrt(d)':>18}")
for dimension in [2, 5, 10, 50, 100, 500, 1_000]:
    cosines = sample_cosines(dimension, pair_count=10_000)
    angles = np.degrees(np.arccos(np.clip(cosines, -1, 1)))
    predicted = np.degrees(1 / np.sqrt(dimension))
    print(f"{dimension:>10} {angles.std():>20.2f} {predicted:>18.2f}")

### Nearly orthogonal is not exactly orthogonal

A $d$-dimensional space can contain at most $d$ mutually orthogonal nonzero vectors. Nevertheless, it can contain *many more* vectors whose pairwise dot products are small. This distinction explains both the surprising capacity of high-dimensional spaces and the optimization code from the screenshot.

## What the screenshot code is doing

The screenshot creates a matrix $X$ with one normalized vector per row. Its central operation is:

```python
dot_products = big_matrix @ big_matrix.T
diff = dot_products - torch.eye(num_vectors)
loss = (diff.abs() - dot_diff_cutoff).relu().sum()
loss += num_vectors * diff.diag().pow(2).sum()
```

`dot_products` is the **Gram matrix** $G = XX^\top$:

- $G_{ii}$ is the squared length of vector $i$ and should be 1.
- $G_{ij}$ is the dot product between different vectors and should be near 0.
- Therefore an ideal orthonormal collection would have $G = I$.

Subtracting the identity makes the desired value zero everywhere. The ReLU term ignores off-diagonal dot products within a small tolerance and penalizes larger ones. The diagonal penalty keeps vector lengths near 1. `loss.backward()` calculates how every coordinate affects that penalty, and Adam takes a small step that reduces it. Repeating this process *nudges* vectors away from directions to which they are too similar or too opposite.

The original choice—10,000 vectors in 100 dimensions—cannot produce a Gram matrix equal to the 10,000 × 10,000 identity: $G$ has rank at most 100. It also creates 100 million dot products per step, so the example below uses much smaller values.

## A transparent NumPy version of the nudge

We use the same hinge-like rule as the screenshot. For every pair with $|x_i \cdot x_j|$ above the cutoff, we calculate a direction that reduces that absolute dot product. We then renormalize the rows. This is plain gradient descent rather than Adam, but the geometric idea is the same.

In [ ]:
def pairwise_cosines(vectors):
    gram = vectors @ vectors.T
    off_diagonal = ~np.eye(len(vectors), dtype=bool)
    return gram[off_diagonal]


def nudge_apart(vectors, cutoff=0.25, steps=600, learning_rate=20.0):
    vectors = vectors.copy()
    losses = []
    pair_count = len(vectors) * (len(vectors) - 1)

    for _ in range(steps):
        gram = vectors @ vectors.T
        active = np.abs(gram) > cutoff
        np.fill_diagonal(active, False)

        # Subgradient of sum(max(abs(dot) - cutoff, 0)).
        gram_gradient = np.sign(gram) * active
        vector_gradient = 2 * (gram_gradient @ vectors) / pair_count
        vectors -= learning_rate * vector_gradient
        vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

        violations = np.maximum(np.abs(gram) - cutoff, 0)
        np.fill_diagonal(violations, 0)
        losses.append(violations.sum() / pair_count)

    return vectors, losses

In [ ]:
vector_count = 80
dimension = 20
cutoff = 0.25

before = random_unit_vectors(vector_count, dimension, rng)
after, losses = nudge_apart(before, cutoff=cutoff)

before_cosines = pairwise_cosines(before)
after_cosines = pairwise_cosines(after)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses)
axes[0].set(title="Optimization loss", xlabel="step", ylabel="mean excess above cutoff")
axes[1].hist(np.degrees(np.arccos(np.clip(before_cosines, -1, 1))), bins=50, alpha=0.6, label="before")
axes[1].hist(np.degrees(np.arccos(np.clip(after_cosines, -1, 1))), bins=50, alpha=0.6, label="after")
axes[1].axvline(90, color="black", linestyle="--", linewidth=1)
axes[1].set(title="Pairwise angles", xlabel="angle (degrees)", ylabel="count")
axes[1].legend()
fig.tight_layout()

In [ ]:
def cosine_summary(values):
    absolute = np.abs(values)
    return {
        "mean absolute cosine": absolute.mean(),
        "95th percentile": np.quantile(absolute, 0.95),
        "maximum absolute cosine": absolute.max(),
    }


print("Before:", cosine_summary(before_cosines))
print("After: ", cosine_summary(after_cosines))

### What changed?

The optimization mainly trims the tails: pairs with unusually small or large angles are pushed toward $90^\circ$. The mean absolute cosine may not improve much because spreading many more than $d$ vectors evenly requires tradeoffs.

There is a mathematical floor on how small all pairwise correlations can be. The **Welch bound** implies that for $n$ unit vectors in $d$ dimensions, the largest absolute pairwise dot product cannot be smaller than

$$
\sqrt{\frac{n-d}{d(n-1)}} \quad \text{when } n>d.
$$

This is a lower bound, not a promise that the optimizer will reach it.

In [ ]:
welch_bound = np.sqrt((vector_count - dimension) / (dimension * (vector_count - 1)))
print(f"Welch lower bound: {welch_bound:.3f}")
print(f"Chosen cutoff:    {cutoff:.3f}")
print(f"Achieved maximum: {np.max(np.abs(after_cosines)):.3f}")

## The Johnson–Lindenstrauss lemma

The Johnson–Lindenstrauss (JL) lemma asks a complementary question: **Can we move a finite collection of high-dimensional points into fewer dimensions without badly changing their pairwise distances?**

It says that for $n$ points and distortion tolerance $0 < \varepsilon < 1$, there is a mapping into roughly

$$
m = O\left(\frac{\log n}{\varepsilon^2}\right)
$$

dimensions that preserves every squared pairwise distance up to a factor of $1 \pm \varepsilon$. Crucially, the required target dimension depends logarithmically on the number of points, not on the original dimension.

A simple construction is a random projection: multiply every point by a matrix with independent Gaussian entries scaled by $1/\sqrt{m}$. High-dimensional concentration makes the projection preserve lengths and dot products with high probability. That is the same concentration phenomenon behind random vectors being nearly orthogonal.

In [ ]:
point_count = 200
original_dimension = 500
points = rng.normal(size=(point_count, original_dimension))
pair_indices = np.triu_indices(point_count, k=1)
original_squared_distances = (
    (points[:, None, :] - points[None, :, :]) ** 2
).sum(axis=2)[pair_indices]

print(f"{'target dimension':>18} {'median distortion':>20} {'95th percentile':>18}")
for target_dimension in [10, 30, 100, 300]:
    projection = rng.normal(
        scale=1 / np.sqrt(target_dimension),
        size=(original_dimension, target_dimension),
    )
    projected = points @ projection
    projected_squared_distances = (
        (projected[:, None, :] - projected[None, :, :]) ** 2
    ).sum(axis=2)[pair_indices]
    distortion = np.abs(projected_squared_distances / original_squared_distances - 1)
    print(
        f"{target_dimension:>18} "
        f"{np.median(distortion):>20.3f} "
        f"{np.quantile(distortion, 0.95):>18.3f}"
    )

The experiment shows the trend rather than the lemma's worst-case guarantee: more target dimensions produce a tighter concentration of distance distortions around zero. A new random projection gives slightly different numbers each time.

## Takeaways

- Independent random vectors in $d$ dimensions meet at about $90^\circ$, with angular spread proportional to $1/\sqrt{d}$.
- Only $d$ vectors can be exactly mutually orthogonal in $d$ dimensions, but many more can be approximately orthogonal.
- The screenshot minimizes large entries of a Gram matrix, thereby nudging overly aligned or anti-aligned vector pairs toward orthogonality.
- With more vectors than dimensions, the optimizer must compromise; it cannot make the Gram matrix equal to the identity.
- The Johnson–Lindenstrauss lemma uses the same high-dimensional concentration phenomenon to preserve distances under random dimensionality reduction.